In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from dataloader import load_train, load_val
from sklearn.preprocessing import StandardScaler
import numpy as np
import pyarrow.parquet as pq

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

In [ ]:
#check data names -> renamed dataloaders segment_id to segment

path = "/work/data_splits/train.parquet"

try:
    pf = pq.ParquetFile(path)
    print("Row groups:", pf.num_row_groups)
    print("Schema:", pf.schema_arrow)
except Exception as e:
    print("ParquetFile error:", repr(e))


In [ ]:



class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=5000):
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        pos = torch.arange(0, max_len).unsqueeze(1).float()
        div = torch.exp(torch.arange(0, d_model, 2).float()
                        * (-torch.log(torch.tensor(10000.0)) / d_model))
        pe[:, 0::2] = torch.sin(pos * div)
        pe[:, 1::2] = torch.cos(pos * div)
        pe = pe.unsqueeze(1)
        self.register_buffer('pe', pe)

    def forward(self, x):
        return x + self.pe[:x.size(0)]
    

class TrajectoryTransformer30to10(nn.Module):
    def __init__(
        self,
        input_dim: int,      # e.g. lat, lon, sog, cog
        output_dim: int,     # same as input dim
        d_model: int = 128,
        nhead: int = 4,
        num_layers: int = 3,
        dim_feedforward: int = 256,
    ):
        super().__init__()

        self.input_proj = nn.Linear(input_dim, d_model) 
        self.pos_enc = PositionalEncoding(d_model)

        enc_layer = nn.TransformerEncoderLayer( #one layer for now
            d_model=d_model,
            nhead=nhead,
            dim_feedforward=dim_feedforward,
            batch_first=False
        )

        self.encoder = nn.TransformerEncoder(enc_layer, num_layers)

        # Output: 10 future steps
        self.out = nn.Linear(d_model, output_dim * 10)

        self.output_dim = output_dim

    def forward(self, src):
        """
        src: [batch, 30, input_dim]
        return: [batch, 10, output_dim]
        """

        x = self.input_proj(src)      # [B, 30, d_model]
        x = x.transpose(0, 1)         # [30, B, d_model]

        x = self.pos_enc(x)
        x = self.encoder(x)

        # take last timestep: index -1 = last of 30
        last_state = x[-1]            # [B, d_model]

        out = self.out(last_state)    # [B, output_dim*10]

        # reshape to [B, 10, output_dim]
        return out.reshape(-1, 10, self.output_dim)


In [ ]:

train_df = load_train()
#val_ds   = load_val()


In [ ]:


# unpack tensors
X_train, Y_train = train_df.tensors   # shapes: [N,30,4] and [N,10,4]

# convert to numpy for sklearn
X_np = X_train.reshape(-1, X_train.shape[-1]).numpy()   # shape: [N*30, 4]
Y_np = Y_train.reshape(-1, Y_train.shape[-1]).numpy()   # shape: [N*10, 4]


# ---- FIT SCALER ONLY ON INPUT X ----
scaler = StandardScaler()
scaler.fit(X_np)

# store for inference later
import joblib
joblib.dump(scaler, "scaler.pkl")
print("Scaler fitted:", scaler.mean_, scaler.scale_)


In [ ]:
# transform
X_scaled = scaler.transform(X_np).reshape(X_train.shape)
Y_scaled = scaler.transform(Y_np).reshape(Y_train.shape)

# convert back to tensors
X_scaled = torch.tensor(X_scaled, dtype=torch.float32)
Y_scaled = torch.tensor(Y_scaled, dtype=torch.float32)

# build scaled dataset
train_ds_scaled = torch.utils.data.TensorDataset(X_scaled, Y_scaled)


In [ ]:
#train_loader = torch.utils.data.DataLoader(
    #train_df, batch_size=64, shuffle=True
#)
'''val_loader = torch.utils.data.DataLoader(
    val_ds, batch_size=64, shuffle=False
)'''
train_loader = torch.utils.data.DataLoader(
    train_ds_scaled, batch_size=64, shuffle=True
)


In [ ]:
def mse(pred, target):
    return F.mse_loss(pred, target, reduction='mean')

def rmse(pred, target):
    return torch.sqrt(F.mse_loss(pred, target, reduction='mean'))

def mae(pred, target):
    return F.l1_loss(pred, target, reduction='mean')

In [ ]:
num_epochs = 15
model = TrajectoryTransformer30to10(
    input_dim=4,      # features dim from dataloader
    output_dim=4,     # predicting same set of features
    d_model=128,
    nhead=4,
    num_layers=3,
    dim_feedforward=256
).to(device)


In [ ]:
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)

mse  = torch.nn.MSELoss()
mae  = torch.nn.L1Loss()

# RMSE is not a built-in loss; define as a function -> maybe not needed...
def rmse(pred, target):
    return torch.sqrt(mse(pred, target))



In [ ]:
train_loader = torch.utils.data.DataLoader(
    train_ds_scaled,
    batch_size=64,
    shuffle=True
)


In [ ]:
for epoch in range(num_epochs):
    model.train()

    train_mse  = 0.0
    train_rmse = 0.0
    train_mae  = 0.0

    for X, Y in train_loader:
        X, Y = X.to(device), Y.to(device)

        optimizer.zero_grad()
        pred = model(X)

        # Backprop loss = MSE
        loss = mse(pred, Y)
        loss.backward()
        optimizer.step()

        # --- METRICS ---
        mse_val  = loss.item()               # reuse computed loss
        rmse_val = mse_val ** 0.5            # faster than sqrt() in torch
        mae_val  = mae(pred, Y).item()

        train_mse  += mse_val
        train_rmse += rmse_val
        train_mae  += mae_val

    # Averages
    train_mse  /= len(train_loader)
    train_rmse /= len(train_loader)
    train_mae  /= len(train_loader)

    print(
        f"Epoch {epoch}: "
        f"MSE={train_mse:.4f}, "
        f"RMSE={train_rmse:.4f}, "
        f"MAE={train_mae:.4f}"
    )

